In [3]:
import cv2

def check_camera_ratio(device_index=8):
    # 打开摄像头
    cap = cv2.VideoCapture(device_index)
    
    if not cap.isOpened():
        print(f"错误：无法打开设备 /dev/video{device_index}")
        return

    # 获取宽度和高度
    width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    
    if width > 0 and height > 0:
        aspect_ratio = width / height
        print(f"--- 设备信息 /dev/video{device_index} ---")
        print(f"分辨率: {int(width)}x{int(height)}")
        print(f"宽高比: {aspect_ratio:.2f}")
        
        # 常见比例识别
        if round(aspect_ratio, 2) == 1.78:
            print("识别类型: 16:9 (宽屏)")
        elif round(aspect_ratio, 2) == 1.33:
            print("识别类型: 4:3 (标准)")
        else:
            print("识别类型: 自定义比例")
    else:
        print("无法获取分辨率信息。")

    # 释放资源
    cap.release()

if __name__ == "__main__":
    check_camera_ratio(0)
    print()
    check_camera_ratio(3)

--- 设备信息 /dev/video0 ---
分辨率: 640x480
宽高比: 1.33
识别类型: 4:3 (标准)

--- 设备信息 /dev/video3 ---
分辨率: 640x480
宽高比: 1.33
识别类型: 4:3 (标准)


In [7]:
import cv2
import numpy as np

def run_pip_demo():
    # 1. 打开两个摄像头
    cap0 = cv2.VideoCapture(0)
    cap3 = cv2.VideoCapture(3)

    if not cap0.isOpened() or not cap3.isOpened():
        print("错误：无法打开 video0 或 video3")
        return

    print("正在运行... 按 'q' 键退出")

    while True:
        ret0, frame0 = cap0.read()
        ret3, frame3 = cap3.read()

        if not ret0 or not ret3:
            break

        # --- A. 调整 video0 的亮度和对比度 ---
        # alpha: 对比度 (1.0-3.0), beta: 亮度 (0-100)
        # 这里使用 cv2.convertScaleAbs 效率最高
        alpha = 1.5  # 对比度增加 20%
        beta = 30    # 亮度增加 30
        frame0 = cv2.convertScaleAbs(frame0, alpha=alpha, beta=beta)

        # --- B. 缩小 video3 的画面 ---
        # 目标尺寸 240x180 (注意 OpenCV 格式是 (宽, 高))
        small_frame3 = cv2.resize(frame3, (240, 180))

        # --- C. 将 video3 叠加到 video0 的右下角 ---
        h0, w0, _ = frame0.shape
        h3, w3, _ = small_frame3.shape

        # 计算起始坐标
        x_offset = w0 - w3 - 10  # 距离右边缘 10 像素
        y_offset = h0 - h3 - 10  # 距离下边缘 10 像素

        # 覆盖区域 (ROI)
        frame0[y_offset:y_offset+h3, x_offset:x_offset+w3] = small_frame3

        # --- D. 显示结果 ---
        cv2.imshow('Camera PIP Mode', frame0)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap0.release()
    cap3.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_pip_demo()

正在运行... 按 'q' 键退出


In [ ]:
import cv2
import os
import time

def run_pip_recorder():
    save_path = "/home/cuhk/Documents/visionpro-kinova-rl/hil-serl/testlx/camera/output"
    if not os.path.exists(save_path):
        os.makedirs(save_path)

    cap0 = cv2.VideoCapture(0)
    cap3 = cv2.VideoCapture(3)

    recording = False
    out = None

    print("--- 控制说明 ---")
    print("按下 's' 键：开始/停止录制")
    print("按下 'q' 键：退出程序")

    while True:
        ret0, frame0 = cap0.read()
        ret3, frame3 = cap3.read()

        if not ret0 or not ret3:
            print("警告：无法从摄像头获取画面")
            break

        # 1. 调整 video0 的亮度和对比度
        # alpha: 对比度 (1.2), beta: 亮度 (30)
        proc_frame0 = cv2.convertScaleAbs(frame0, alpha=1.2, beta=30)

        # 2. 缩小 video3 并叠加到右下角
        small_frame3 = cv2.resize(frame3, (240, 180))
        h0, w0, _ = proc_frame0.shape
        h3, w3, _ = small_frame3.shape
        
        x_offset, y_offset = w0 - w3 - 10, h0 - h3 - 10
        proc_frame0[y_offset:y_offset+h3, x_offset:x_offset+w3] = small_frame3

        # 3. 录制逻辑
        if recording:
            if out is not None:
                out.write(proc_frame0)
            # 在预览画面上打个红点，表示正在录制
            cv2.circle(proc_frame0, (30, 30), 10, (0, 0, 255), -1)

        # 4. 显示画面
        cv2.imshow('PIP Recorder', proc_frame0)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('s'):
            if not recording:
                # 开始录制：初始化 VideoWriter
                timestamp = time.strftime("%Y%m%d-%H%M%S")
                file_name = os.path.join(save_path, f"output_{timestamp}.mp4")
                fourcc = cv2.VideoWriter_fourcc(*'mp4v') # 使用 mp4 编码
                fps = 20.0  # 建议根据实际运行帧率调整
                out = cv2.VideoWriter(file_name, fourcc, fps, (w0, h0))
                recording = True
                print(f"开始录制: {file_name}")
            else:
                # 停止录制
                recording = False
                if out:
                    out.release()
                print("录制已停止并保存。")

    cap0.release()
    cap3.release()
    if out:
        out.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_pip_recorder()